Exploratory data analysis (EDA) is a crucial component of data science which allows you to understand the basics of what your data looks like and what kinds of questions might be answered. For this task, we are going to clean, sanitise and explore our data. Using the automobile dataset, complete the steps outlined in this notebook by writing code in the cells.

In [ ]:
# Import libraries

import numpy as np
import pandas as pd
import seaborn as sns

from datetime import datetime
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Load the automobile dataset
try:
    automobiles_df = pd.read_csv('automobile.txt')
    print(f"Dataset loaded successfully!")
    print(f"Dataset shape: {automobiles_df.shape}")
    print(f"\nFirst few rows:")
    print(automobiles_df.head())
except FileNotFoundError:
        print(f"Error: File 'automobile.txt' not found.")


In [ ]:
print(f"{automobiles_df.columns.tolist()}")

### Data Cleansing
#### Clean the data

Identify columns that are redundant or unnecessary. It is always easier to make your decisions based on data which is relevant and concise. Remove the following columns `['normalized-losses', 'symboling']` from the data set as they will not be used in the analysis.

In [ ]:
def remove_redundant_columns(df):
    """
    Remove columns that are not useful for analysis.
    Removes: 'normalized-losses' (contains many missing values)
             'symboling' (risk rating, not relevant to analysis)
    """
    columns_to_drop = ['normalized-losses', 'symboling']
    df = df.drop(columns=columns_to_drop)
    
    print("Redundant columns removed.")
    print(f"Remaining columns: {df.columns.tolist()}")
    print(f"Dataset shape: {df.shape}\n")
    
    return df

automobiles_df = remove_redundant_columns(automobiles_df)
automobiles_df.head()

#### Remove any duplicate rows

In [ ]:
# Code here
def remove_duplicate_rows(df):
    """
    Identify and remove duplicate rows from the dataset.
    """
    duplicate_count = df.duplicated().sum()
    print(f"Number of duplicate rows found: {duplicate_count}")
    
    df = df.drop_duplicates()
    print(f"Dataset shape after removing duplicates: {df.shape}\n")
    
    return df

# Remove duplicate rows
automobiles_df = remove_duplicate_rows(automobiles_df)

#### Remove rows with missing data

Some automobiles in the database have missing values which implies that their values have not been recorded or some information is missing. Discard such entries from the dataframe.

In [ ]:
def remove_missing_data(df):
    """
    Remove rows with missing values (represented as '?').
    
    """
    # Replace '?' with NaN for easier handling
    df = df.replace('?', np.nan)
    
    # Display missing data before removal
    print("Missing values (represented as '?') before cleaning:")
    missing_summary = df.isnull().sum()
    print(missing_summary[missing_summary > 0])
    
    print(f"\nRows with missing values: {df.isnull().any(axis=1).sum()}")
    print(f"Total rows before removing missing values: {len(df)}")
    
    # Remove all rows with any missing values
    df = df.dropna()
    
    print(f"Total rows after removing missing values: {len(df)}")
    print(f"Dataset shape after data cleansing: {df.shape}\n")
    
    return df

# Remove missing data
automobiles_df = remove_missing_data(automobiles_df)

Change columns with numerical data column to an integer data type using numpy’s `int64` method.

In [ ]:
def convert_data_types(df):
    """
    Convert numerical columns to appropriate data types.
    Converts specified columns to int64 and float64 as appropriate.
    """
    print("Data types before conversion:")
    print(df.dtypes)
    # Define columns that should be converted to integer (excluding num-of-doors)
    integer_columns = ['engine-size', 
                       'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price', 'curb-weight']
    
    # Convert specified columns to int64
    for col in integer_columns:
        if col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('int64')
            except Exception as e:
                print(f"Warning: Could not convert {col}: {e}")
    
    # Convert float columns
    float_columns = ['wheel-base', 'length', 'width', 'height', 'bore', 'stroke', 'compression-ratio']
    for col in float_columns:
        if col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
            except Exception as e:
                print(f"Warning: Could not convert {col}: {e}")
    
    print("\nData types after conversion:")
    print(df.dtypes)
    print()
    
    return df

# Convert data types
automobiles_df = convert_data_types(automobiles_df)

### Finding Certain Categories
Locate all automobiles in the "hatchback" genre.

In [ ]:
# Create a dataframe with all the cars in the "hatchback' category
# hatchback_cars =
def find_hatchback_cars(df):
    """
    Find and display all hatchback vehicles in the dataset.
    """
    
    hatchback_cars = df[df['body-style'] == 'hatchback']
    
    print(f"\nNumber of hatchback cars in dataset: {len(hatchback_cars)}")
    print(f"\nFirst 10 hatchback cars:")
    print(hatchback_cars[['make', 'body-style', 'price', 'horsepower', 'city-mpg']].head(10))
    print()
    
    return hatchback_cars

hatchback_cars = find_hatchback_cars(automobiles_df)

In [ ]:
def display_descriptive_statistics(df):
    """
    Display comprehensive statistical summary of the dataset.
    """
    print("=" * 70)
    print("DESCRIPTIVE STATISTICS")
    print("=" * 70)
    
    print("\nStatistical Summary:")
    print(df.describe())
    
    print("\nDataset Information:")
    print(df.info())
    print()

display_descriptive_statistics(automobiles_df)

### Now onto the exploration

#### Identify relationships between variables (features)

The main goal here is to identify and create relationships that can help you formulate ideas. We have defined questions to help you identify some relationships to explore.

#### Which are the 5 most expensive cars?

How do the most expensive and cheapest cars compare? Exploring the most expensive cars highlights if some moviecars are worth the money spent on them based on their fuel economy (mpg or miles per gallon)

In [ ]:
def analyze_most_expensive_cars(df):
    """
    Analyze the 5 most expensive cars and compare with the cheapest 5.
    
    """
    print("=" * 70)
    print("MOST EXPENSIVE vs CHEAPEST CARS ANALYSIS")
    print("=" * 70)
    
    # Find the 5 most expensive cars
    top_5_expensive = df.nlargest(5, 'price')[['make', 'price', 'city-mpg', 'highway-mpg', 'horsepower', 'engine-size']]
    
    print("\nTop 5 Most Expensive Cars:")
    print(top_5_expensive.to_string())
    
    # Calculate average MPG for these cars
    top_5_expensive['avg-mpg'] = (top_5_expensive['city-mpg'] + top_5_expensive['highway-mpg']) / 2
    print(f"\nAverage MPG for top 5 most expensive cars: {top_5_expensive['avg-mpg'].mean():.2f}")
    
    # Compare with the 5 cheapest cars
    top_5_cheapest = df.nsmallest(5, 'price')[['make', 'price', 'city-mpg', 'highway-mpg', 'horsepower', 'engine-size']]
    print("\nTop 5 Cheapest Cars:")
    print(top_5_cheapest.to_string())
    
    top_5_cheapest['avg-mpg'] = (top_5_cheapest['city-mpg'] + top_5_cheapest['highway-mpg']) / 2
    print(f"\nAverage MPG for top 5 cheapest cars: {top_5_cheapest['avg-mpg'].mean():.2f}")
    print()

analyze_most_expensive_cars(automobiles_df)


#### Which manufacturer builds the most fuel efficient vehicles?

Compare the average mpg for each vehicle manufacture's vehicles and create a bar plot

In [ ]:
def analyze_manufacturer_fuel_efficiency(df):
    """
    Analyze and visualize fuel efficiency by manufacturer using seaborn.
    """
    print("=" * 70)
    print("MANUFACTURER FUEL EFFICIENCY ANALYSIS")
    print("=" * 70)
    
    # Calculate average city and highway MPG for each manufacturer
    manufacturer_mpg = df.groupby('make')[['city-mpg', 'highway-mpg']].mean().round(2)
    manufacturer_mpg['avg-mpg'] = ((manufacturer_mpg['city-mpg'] + manufacturer_mpg['highway-mpg']) / 2).round(2)
    manufacturer_mpg = manufacturer_mpg.sort_values('avg-mpg', ascending=False)
    
    print("\nAverage Fuel Economy by Manufacturer (sorted by efficiency):")
    print(manufacturer_mpg)
    
    print(f"\nMost Fuel-Efficient Manufacturer: {manufacturer_mpg.index[0]} with {manufacturer_mpg['avg-mpg'].iloc[0]} MPG average")
    print(f"Least Fuel-Efficient Manufacturer: {manufacturer_mpg.index[-1]} with {manufacturer_mpg['avg-mpg'].iloc[-1]} MPG average")
    print()
    
    # Create visualization
    print("Creating fuel efficiency visualization...")
    
    # Reset index to convert index to column for sns.barplot
    manufacturer_mpg_reset = manufacturer_mpg.reset_index()
    
    # Sort in ascending order for better visualization
    manufacturer_mpg_reset = manufacturer_mpg_reset.sort_values('avg-mpg', ascending=True)
    
    # Set figure size dynamically based on number of manufacturers
    fig_height = max(8, len(manufacturer_mpg_reset) * 0.3)
    plt.figure(figsize=(12, fig_height))
    
    # Create seaborn bar plot
    ax = sns.barplot(
        data=manufacturer_mpg_reset,
        y='make',
        x='avg-mpg',
        hue='make',
        legend=False,
        palette='Blues_r',  # Gradient from darkest (most efficient) to lightest
        edgecolor='black',
        linewidth=0.5
    )
    
    # Get max value for positioning labels
    max_mpg = manufacturer_mpg_reset['avg-mpg'].max()
    
    # Add value labels on bars
    for i, bar in enumerate(ax.patches):
        width = bar.get_width()
        ax.text(
            width + (max_mpg * 0.01),
            bar.get_y() + bar.get_height() / 2,
            f'{width:.1f}',
            va='center',
            ha='left',
            fontsize=9,
            fontweight='bold'
        )
    
    # Styling
    ax.set_xlim(0, max_mpg * 1.12)  # Leave margin for data labels
    ax.set_xlabel('Average MPG (Combined City + Highway)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Vehicle Manufacturer', fontsize=12, fontweight='bold')
    ax.set_title(
        'Average Fuel Economy by Automobile Manufacturer',
        fontsize=14,
        fontweight='bold',
        pad=15
    )
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Insight: Chevrolet leads in fuel efficiency in this dataset, while Jaguar and Porsche are least efficient.\n")


# Run the function
analyze_manufacturer_fuel_efficiency(automobiles_df)

#### Which vehicles have the largest engine capacity.
Sort the dataframe based on the engine-size column.

In [ ]:
def analyze_engine_size(df):
    """
    Analyze vehicles by engine size and visualize relationships.
    """
    print("=" * 70)
    print("ENGINE SIZE ANALYSIS")
    print("=" * 70)
    
    # Find largest engines
    largest_engines = df.nlargest(10, 'engine-size')[['make', 'engine-size', 'horsepower', 'price', 'city-mpg']]
    
    print("\nTop 10 Vehicles with Largest Engine Capacity:")
    print(largest_engines.to_string())
    
    # Calculate statistics
    print(f"\nEngine Size Statistics:")
    print(f"Largest engine: {df['engine-size'].max()} cubic inches")
    print(f"Smallest engine: {df['engine-size'].min()} cubic inches")
    print(f"Average engine size: {df['engine-size'].mean():.2f} cubic inches")
    print(f"Median engine size: {df['engine-size'].median():.2f} cubic inches")
    print()
    
    # Create visualizations
    print("Creating engine size visualizations...")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram of engine sizes
    axes[0].hist(df['engine-size'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Engine Size (cubic inches)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Number of Vehicles', fontsize=12, fontweight='bold')
    axes[0].set_title('Distribution of Engine Sizes', fontsize=14, fontweight='bold')
    axes[0].axvline(df['engine-size'].mean(), color='red', linestyle='--', linewidth=2, 
                    label=f"Mean: {df['engine-size'].mean():.0f}")
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Scatter plot: Engine Size vs Horsepower
    axes[1].scatter(df['engine-size'], df['horsepower'], alpha=0.6, color='coral')
    axes[1].set_xlabel('Engine Size (cubic inches)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Horsepower', fontsize=12, fontweight='bold')
    axes[1].set_title('Engine Size vs Horsepower', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate correlation
    correlation = df['engine-size'].corr(df['horsepower'])
    print(f"\nCorrelation between Engine Size and Horsepower: {correlation:.3f}")
    print("Insight: There is a strong positive correlation between engine size and horsepower.\n")

analyze_engine_size(automobiles_df)

#### Which vehicle manufacturer has the most car models in the dataset

In [ ]:
# Code here
def analyze_manufacturer_models(df):
    """
    Analyze which manufacturer has the most car models in the dataset.
    
    """
    print("=" * 70)
    print("MANUFACTURER MODEL COUNT ANALYSIS")
    print("=" * 70)
    
    # Count the number of models for each manufacturer
    models_per_manufacturer = df['make'].value_counts().sort_values(ascending=False)
    
    print("\nNumber of Car Models per Manufacturer:")
    print(models_per_manufacturer)
    
    print(f"\nManufacturer with most models: {models_per_manufacturer.index[0]} with {models_per_manufacturer.iloc[0]} models")
    print(f"Total number of manufacturers: {len(models_per_manufacturer)}")
    print()
    
    # Create visualization
    print("Creating manufacturer model count visualization...")
    
    plt.figure(figsize=(14, 6))
    
    bars = plt.bar(range(len(models_per_manufacturer)), models_per_manufacturer.values, color='mediumseagreen')
    
    # Highlight the top manufacturer
    bars[0].set_color('darkgreen')
    
    plt.xticks(range(len(models_per_manufacturer)), models_per_manufacturer.index, rotation=45, ha='right')
    plt.xlabel('Manufacturer', fontsize=12, fontweight='bold')
    plt.ylabel('Number of Models', fontsize=12, fontweight='bold')
    plt.title('Number of Car Models by Manufacturer in Dataset', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(models_per_manufacturer.values):
        plt.text(i, v + 0.1, str(v), ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("Insight: Toyota and Nissan dominate the dataset with the most car models.\n")

analyze_manufacturer_models(automobiles_df)